# Data Integration Pipeline: TechTrove E-Commerce
### From Disparate Raw Sources to Analytical Star Schema & Data Quality Lineage
**Course:** Data Warehousing / Data Engineering  
**Role:** Data Engineer  
**Organization:** TechTrove E-Commerce  
**Deliverables:**
1. Cleaned & Aligned Star Schema (`dim_customer.csv`, `dim_product.csv`, `fact_sales.csv`)
2. Audit-ready `data_quality_report.csv`
3. Business Sales Summaries (`summary_by_province.csv`, `summary_by_category.csv`)
4. Data Integrity Assertions & Quality Funnel Visualization (Challenge +2)
5. Comprehensive Answers to 6 Analytical Questions


## 1. Import Libraries & Set Up Working Directory


In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Paths
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'output'
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Container for tracking Data Quality anomalies across all ETL stages
dq_issues = []
print(f"Working directory: {BASE_DIR}")
print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


## 2. Extract & Profile Raw Datasets (Step 1)
Extract raw data across CSV, Excel, and nested JSON files and profile initial data quality (shapes, data types, nulls, and duplicate records).


In [ ]:
# Extract raw data
df_o1_raw = pd.read_csv(DATA_DIR / 'orders_2026_01.csv')
df_o2_raw = pd.read_csv(DATA_DIR / 'orders_2026_02.csv')
df_c_raw = pd.read_csv(DATA_DIR / 'customers_crm.csv')
df_p_raw = pd.read_excel(DATA_DIR / 'product_master.xlsx')

with open(DATA_DIR / 'payments.json', 'r', encoding='utf-8') as f:
    pay_json = json.load(f)
df_pay_raw = pd.json_normalize(pay_json)

# Raw Data Profiling Summary
raw_datasets = {
    'orders_2026_01.csv': df_o1_raw,
    'orders_2026_02.csv': df_o2_raw,
    'customers_crm.csv': df_c_raw,
    'product_master.xlsx': df_p_raw,
    'payments.json': df_pay_raw
}

profile_list = []
for name, df in raw_datasets.items():
    profile_list.append({
        'Source File': name,
        'Rows': len(df),
        'Columns': len(df.columns),
        'Null Values': int(df.isnull().sum().sum()),
        'Duplicate Rows': int(df.duplicated().sum()),
        'Column Names': list(df.columns)
    })

pd.DataFrame(profile_list)


## 3. Schema Alignment & Combining Orders (Step 2)
Address schema drift between monthly order files:
- Rename `ordered_at` $\rightarrow$ `order_date`, `qty` $\rightarrow$ `quantity`, `discount_pct` $\rightarrow$ `discount`.
- Convert percentage strings (e.g. `'5%'`) to numeric float values (`0.05`).
- Unify date formats into ISO standard datetime.
- Concatenate using `pd.concat(..., ignore_index=True)`.
- Deduplicate by `order_id` keeping the latest occurrence (`keep='last'`) per business rules.


In [ ]:
# Schema alignment for February orders
df_o2_aligned = df_o2_raw.rename(columns={
    'ordered_at': 'order_date',
    'qty': 'quantity',
    'discount_pct': 'discount'
}).copy()

# Parse discount percentage
df_o2_aligned['discount'] = df_o2_aligned['discount'].astype(str).str.rstrip('%').astype(float) / 100.0

# Standardize datetime formats
df_o1_aligned = df_o1_raw.copy()
df_o1_aligned['order_date'] = pd.to_datetime(df_o1_aligned['order_date'])
df_o2_aligned['order_date'] = pd.to_datetime(df_o2_aligned['order_date'], format='%d/%m/%Y %H:%M')

# Combine orders
df_orders_combined = pd.concat([df_o1_aligned, df_o2_aligned], ignore_index=True)
print(f"Total combined orders: {len(df_orders_combined)} rows (Jan: {len(df_o1_aligned)} + Feb: {len(df_o2_aligned)})")

# Track duplicates
dup_orders = df_orders_combined[df_orders_combined.duplicated(subset=['order_id'], keep=False)]
for oid in dup_orders['order_id'].unique():
    rows = dup_orders[dup_orders['order_id'] == oid]
    dq_issues.append({
        'stage': '2. Combine & Deduplicate',
        'dataset': 'orders',
        'record_id': oid,
        'column_name': 'order_id',
        'issue_type': 'DUPLICATE_KEY',
        'issue_description': f'Duplicate order_id found ({len(rows)} occurrences). Kept latest record.',
        'action_taken': "DEDUPLICATED (keep='last')"
    })

# Deduplicate orders
df_orders_dedup = df_orders_combined.drop_duplicates(subset=['order_id'], keep='last').copy()
print(f"Orders after deduplication (keep='last'): {len(df_orders_dedup)} rows (Removed {len(df_orders_combined) - len(df_orders_dedup)} duplicates)")


## 4. Data Cleaning & Standardization (Step 3)
- **Customers**: Trim whitespaces, lowercase email addresses, standardize province names (handling English names like `Bangkok`, `Chonburi`, `Chiang Mai`, abbreviations like `กทม.`, and vowel encoding variations like `ขอนเเก่น`). Deduplicate by `customer_id` (`keep='last'`).
- **Products**: Trim text fields, ensure valid active flags.
- **Payments**: Normalize column names (`payment.method` $\rightarrow$ `payment_method`, `payment.status` $\rightarrow$ `payment_status`), trim whitespaces, deduplicate on `order_id` (`keep='last'`).


In [ ]:
# 4.1 Clean Customers CRM
df_c = df_c_raw.copy()
for col in ['customer_id', 'full_name', 'email', 'province', 'signup_date']:
    if col in df_c.columns:
        df_c[col] = df_c[col].astype(str).str.strip()

# Track missing emails
for idx, r in df_c[df_c['email'].isin(['nan', 'none', '', 'NaN']) | df_c['email'].isnull()].iterrows():
    dq_issues.append({
        'stage': '3. Clean Customers',
        'dataset': 'customers',
        'record_id': r['customer_id'],
        'column_name': 'email',
        'issue_type': 'MISSING_EMAIL',
        'issue_description': 'Customer email address is missing/null.',
        'action_taken': 'SET_NULL'
    })
df_c['email'] = df_c['email'].str.lower().replace({'nan': None, 'none': None, '': None})

province_map = {
    'ชลบุรี': 'ชลบุรี', 'Chonburi': 'ชลบุรี', 'ชลบุรี ': 'ชลบุรี',
    'ขอนแก่น': 'ขอนแก่น', 'ขอนเเก่น': 'ขอนแก่น',
    'กรุงเทพมหานคร': 'กรุงเทพมหานคร', 'Bangkok': 'กรุงเทพมหานคร', 'กทม.': 'กรุงเทพมหานคร',
    'ระยอง': 'ระยอง', 'Rayong': 'ระยอง',
    'ภูเก็ต': 'ภูเก็ต', 'Phuket': 'ภูเก็ต',
    'เชียงใหม่': 'เชียงใหม่', 'Chiang Mai': 'เชียงใหม่'
}

for idx, r in df_c.iterrows():
    raw_p = df_c_raw.loc[idx, 'province']
    std_p = province_map.get(raw_p, raw_p)
    if raw_p != std_p:
        dq_issues.append({
            'stage': '3. Clean Customers',
            'dataset': 'customers',
            'record_id': r['customer_id'],
            'column_name': 'province',
            'issue_type': 'NON_STANDARD_PROVINCE',
            'issue_description': f"Non-standard province name: '{raw_p}'",
            'action_taken': f"STANDARDIZED to '{std_p}'"
        })
df_c['province'] = df_c['province'].map(province_map).fillna(df_c['province'])

# Deduplicate customers
dup_c = df_c[df_c.duplicated(subset=['customer_id'], keep=False)]
for cid in dup_c['customer_id'].unique():
    dq_issues.append({
        'stage': '3. Clean Customers',
        'dataset': 'customers',
        'record_id': cid,
        'column_name': 'customer_id',
        'issue_type': 'DUPLICATE_KEY',
        'issue_description': f"Duplicate customer_id '{cid}' in CRM.",
        'action_taken': "DEDUPLICATED (keep='last')"
    })
df_c_dim = df_c.drop_duplicates(subset=['customer_id'], keep='last').copy()
print(f"Cleaned Customer Master: {len(df_c_dim)} unique customers (from {len(df_c_raw)} raw rows)")

# 4.2 Clean Product Master
df_p_dim = df_p_raw.copy()
for col in ['product_id', 'product_name', 'category', 'active_flag']:
    df_p_dim[col] = df_p_dim[col].astype(str).str.strip()
print(f"Cleaned Product Master: {len(df_p_dim)} products")

# 4.3 Clean Payments Gateway
df_pay = df_pay_raw.copy()
df_pay = df_pay.rename(columns={'payment.method': 'payment_method', 'payment.status': 'payment_status'})
for col in ['payment_id', 'order_id', 'payment_method', 'payment_status']:
    if col in df_pay.columns:
        df_pay[col] = df_pay[col].astype(str).str.strip()

dup_pay = df_pay[df_pay.duplicated(subset=['order_id'], keep=False)]
for oid in dup_pay['order_id'].unique():
    dq_issues.append({
        'stage': '3. Clean Payments',
        'dataset': 'payments',
        'record_id': oid,
        'column_name': 'order_id',
        'issue_type': 'DUPLICATE_KEY',
        'issue_description': f"Duplicate payment event for order_id '{oid}'.",
        'action_taken': "DEDUPLICATED (keep='last')"
    })
df_pay_clean = df_pay.drop_duplicates(subset=['order_id'], keep='last').copy()
print(f"Cleaned Payments: {len(df_pay_clean)} unique payment events")


## 5. Multi-Source Integration & Business Validation (Step 4)
- Merge `orders` with `dim_customer` using `pd.merge(..., validate='m:1', indicator='_cust_match')`.
- Merge with `dim_product` using `pd.merge(..., validate='m:1', indicator='_prod_match')`.
- Merge with `payments` using `pd.merge(..., validate='1:1', indicator='_pay_match')`.
- Track referential integrity anomalies, domain errors (`quantity <= 0`, null/invalid `unit_price`), and payment statuses.
- Calculate Net Sales:
  $$\text{net\_sales} = \text{quantity} \times \text{unit\_price} \times (1 - \text{discount})$$


In [ ]:
# Merge steps
m_cust = df_orders_dedup.merge(df_c_dim, on='customer_id', how='left', indicator='_cust_match', validate='m:1')
m_prod = m_cust.merge(df_p_dim, on='product_id', how='left', indicator='_prod_match', validate='m:1')
integrated = m_prod.merge(df_pay_clean, on='order_id', how='left', indicator='_pay_match', validate='1:1')

print(f"Total integrated order rows: {len(integrated)}")
print(f"Customer Match: both={sum(integrated['_cust_match'] == 'both')}, unmatched={sum(integrated['_cust_match'] != 'both')}")
print(f"Product Match:  both={sum(integrated['_prod_match'] == 'both')}, unmatched={sum(integrated['_prod_match'] != 'both')}")
print(f"Payment Match:  both={sum(integrated['_pay_match'] == 'both')}, unmatched={sum(integrated['_pay_match'] != 'both')}")

# Record validation issues
for idx, r in integrated.iterrows():
    oid = r['order_id']
    if r['quantity'] <= 0:
        dq_issues.append({
            'stage': '4. Validate Business Rules',
            'dataset': 'orders',
            'record_id': oid,
            'column_name': 'quantity',
            'issue_type': 'INVALID_QUANTITY',
            'issue_description': f"Quantity <= 0 (value: {r['quantity']})",
            'action_taken': 'EXCLUDED_FROM_FACT_SALES'
        })
    if pd.isnull(r['unit_price']) or r['unit_price'] <= 0:
        dq_issues.append({
            'stage': '4. Validate Business Rules',
            'dataset': 'orders',
            'record_id': oid,
            'column_name': 'unit_price',
            'issue_type': 'INVALID_UNIT_PRICE',
            'issue_description': f"Unit price is null or <= 0 (value: {r['unit_price']})",
            'action_taken': 'EXCLUDED_FROM_FACT_SALES'
        })
    if r['discount'] < 0 or r['discount'] > 1:
        dq_issues.append({
            'stage': '4. Validate Business Rules',
            'dataset': 'orders',
            'record_id': oid,
            'column_name': 'discount',
            'issue_type': 'INVALID_DISCOUNT',
            'issue_description': f"Discount out of range [0, 1] (value: {r['discount']})",
            'action_taken': 'EXCLUDED_FROM_FACT_SALES'
        })
    if r['_cust_match'] != 'both':
        dq_issues.append({
            'stage': '4. Validate Referential Integrity',
            'dataset': 'orders',
            'record_id': oid,
            'column_name': 'customer_id',
            'issue_type': 'UNMATCHED_CUSTOMER_ID',
            'issue_description': f"customer_id '{r['customer_id']}' not found in Customer Master",
            'action_taken': 'EXCLUDED_FROM_FACT_SALES'
        })
    if r['_prod_match'] != 'both':
        dq_issues.append({
            'stage': '4. Validate Referential Integrity',
            'dataset': 'orders',
            'record_id': oid,
            'column_name': 'product_id',
            'issue_type': 'UNMATCHED_PRODUCT_ID',
            'issue_description': f"product_id '{r['product_id']}' not found in Product Master",
            'action_taken': 'EXCLUDED_FROM_FACT_SALES'
        })
    if r['_pay_match'] != 'both':
        dq_issues.append({
            'stage': '4. Validate Payments',
            'dataset': 'orders',
            'record_id': oid,
            'column_name': 'order_id',
            'issue_type': 'UNMATCHED_PAYMENT',
            'issue_description': f"order_id '{oid}' has no matching payment record",
            'action_taken': 'EXCLUDED_FROM_FACT_SALES'
        })
    elif r['payment_status'] != 'PAID':
        dq_issues.append({
            'stage': '4. Validate Business Rules',
            'dataset': 'payments',
            'record_id': oid,
            'column_name': 'payment_status',
            'issue_type': f"PAYMENT_{r['payment_status']}",
            'issue_description': f"Payment status is {r['payment_status']} (not PAID)",
            'action_taken': 'EXCLUDED_FROM_FACT_SALES'
        })

# Filter strictly valid sales transactions
valid_mask = (
    (integrated['quantity'] > 0)
    & (integrated['unit_price'].notnull())
    & (integrated['unit_price'] > 0)
    & (integrated['discount'] >= 0)
    & (integrated['discount'] <= 1)
    & (integrated['_cust_match'] == 'both')
    & (integrated['_prod_match'] == 'both')
    & (integrated['_pay_match'] == 'both')
    & (integrated['payment_status'] == 'PAID')
)

fact_sales = integrated[valid_mask].copy()
fact_sales['net_sales'] = fact_sales['quantity'] * fact_sales['unit_price'] * (1.0 - fact_sales['discount'])

print(f"Valid Sales Transactions (Fact Sales): {len(fact_sales)} rows")
print(f"Excluded Invalid/Unpaid Orders: {len(integrated) - len(fact_sales)} rows")
print(f"Total Net Sales: {fact_sales['net_sales'].sum():,.2f} THB")


## 6. Challenge (+2 Points): Data Integrity Validation & Quality Funnel Chart
Implement `validate_data()` with rigorous assertions, and visualize the record transition from raw orders to verified paid sales.


In [ ]:
def validate_data(df_fact, df_cust, df_prod):
    # 1. Uniqueness
    assert df_fact['order_id'].is_unique, "Assertion Failed: Fact table order_id must be unique!"
    assert df_cust['customer_id'].is_unique, "Assertion Failed: Dimension Customer customer_id must be unique!"
    assert df_prod['product_id'].is_unique, "Assertion Failed: Dimension Product product_id must be unique!"
    
    # 2. Referential Integrity
    assert (~df_fact['customer_id'].isin(df_cust['customer_id'])).sum() == 0, "Foreign key violation on customer_id!"
    assert (~df_fact['product_id'].isin(df_prod['product_id'])).sum() == 0, "Foreign key violation on product_id!"
    
    # 3. Domain & Range Constraints
    assert (df_fact['quantity'] > 0).all(), "Invalid quantity <= 0!"
    assert (df_fact['unit_price'] > 0).all(), "Invalid unit_price <= 0!"
    assert ((df_fact['discount'] >= 0) & (df_fact['discount'] <= 1)).all(), "Invalid discount!"
    assert (df_fact['net_sales'] >= 0).all(), "Invalid negative net_sales!"
    
    # 4. Completeness
    critical_cols = ['order_id', 'order_date', 'customer_id', 'product_id', 'quantity', 'unit_price', 'discount', 'net_sales']
    for col in critical_cols:
        assert df_fact[col].notnull().all(), f"Missing values in critical column: {col}!"
    print("All Data Integrity Assertions Passed 100% Successfully!")

validate_data(fact_sales, df_c_dim, df_p_dim)


In [ ]:
# Data Quality Funnel Visualization
stages = ['1. Raw Orders\n(Jan + Feb)', '2. Deduplicated\nOrders', '3. Valid Master\n& Price/Qty', '4. Fully Paid Sales\n(Fact Sales)']
raw_cnt = len(df_o1_raw) + len(df_o2_raw)
dedup_cnt = len(df_orders_dedup)
matched_valid_cnt = len(integrated[
    (integrated['quantity'] > 0)
    & (integrated['unit_price'].notnull())
    & (integrated['unit_price'] > 0)
    & (integrated['discount'] >= 0)
    & (integrated['discount'] <= 1)
    & (integrated['_cust_match'] == 'both')
    & (integrated['_prod_match'] == 'both')
])
paid_cnt = len(fact_sales)

counts = [raw_cnt, dedup_cnt, matched_valid_cnt, paid_cnt]
retention_pct = [c / raw_cnt * 100 for c in counts]

plt.figure(figsize=(9, 5.5), dpi=150)
colors = ['#4A90E2', '#50E3C2', '#F5A623', '#7ED321']
bars = plt.bar(stages, counts, color=colors, edgecolor='#2C3E50', width=0.55, linewidth=1.5)

for bar, count, pct in zip(bars, counts, retention_pct):
    plt.text(
        bar.get_x() + bar.get_width() / 2.0,
        bar.get_height() + 15,
        f"{count:,} rows\n({pct:.1f}%)",
        ha='center', va='bottom', fontsize=11, fontweight='bold', color='#2C3E50'
    )

plt.ylim(0, max(counts) * 1.22)
plt.title('TechTrove Data Quality Pipeline Funnel', fontsize=15, fontweight='bold', pad=15)
plt.ylabel('Record Count', fontsize=11, fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'data_quality_funnel.png')
plt.show()


## 7. Export Star Schema Dimensions, Fact Table, and DQ Report (Step 5)


In [ ]:
# 1. dim_customer.csv
dim_customer_out = df_c_dim[['customer_id', 'full_name', 'email', 'province', 'signup_date']].copy()
dim_customer_out.to_csv(OUTPUT_DIR / 'dim_customer.csv', index=False, encoding='utf-8-sig')

# 2. dim_product.csv
dim_product_out = df_p_dim[['product_id', 'product_name', 'category', 'standard_price', 'active_flag']].copy()
dim_product_out.to_csv(OUTPUT_DIR / 'dim_product.csv', index=False, encoding='utf-8-sig')

# 3. fact_sales.csv
fact_sales_out = fact_sales[[
    'order_id', 'order_date', 'customer_id', 'product_id',
    'quantity', 'unit_price', 'discount', 'channel',
    'payment_id', 'payment_method', 'net_sales'
]].copy()
fact_sales_out.to_csv(OUTPUT_DIR / 'fact_sales.csv', index=False, encoding='utf-8-sig')

# 4. data_quality_report.csv
df_dq_report = pd.DataFrame(dq_issues)
df_dq_report.to_csv(OUTPUT_DIR / 'data_quality_report.csv', index=False, encoding='utf-8-sig')

# 5. summary_by_province.csv
summary_prov = fact_sales.groupby('province').agg(
    total_orders=('order_id', 'count'),
    total_quantity=('quantity', 'sum'),
    total_net_sales=('net_sales', 'sum')
).reset_index().sort_values(by='total_net_sales', ascending=False)
summary_prov.to_csv(OUTPUT_DIR / 'summary_by_province.csv', index=False, encoding='utf-8-sig')

# 6. summary_by_category.csv
summary_cat = fact_sales.groupby('category').agg(
    total_orders=('order_id', 'count'),
    total_quantity=('quantity', 'sum'),
    total_net_sales=('net_sales', 'sum')
).reset_index().sort_values(by='total_net_sales', ascending=False)
summary_cat.to_csv(OUTPUT_DIR / 'summary_by_category.csv', index=False, encoding='utf-8-sig')

print('All 6 output files successfully created in output/ folder!')


## 8. Summary Aggregations & Sales Analysis (Step 6)


In [ ]:
print('=== SUMMARY BY PROVINCE ===')
display(summary_prov)

print('\n=== SUMMARY BY PRODUCT CATEGORY ===')
display(summary_cat)


## 9. Analytical Answers to Lab Questions (คำตอบคำถามวิเคราะห์ 6 ข้อ)

---

### **คำถามที่ 1: หลังรวมไฟล์ orders มีจำนวนแถวเท่าใด และเหลือกี่แถวหลังลบ duplicate?**
- **คำตอบ:**
  - หลังรวมไฟล์ `orders_2026_01.csv` (361 แถว) และ `orders_2026_02.csv` (391 แถว) ด้วย `pd.concat(..., ignore_index=True)` ได้แถวคำสั่งซื้อรวม **752 แถว**
  - หลังดำเนินการลบ Duplicate โดยใช้ Primary Key `order_id` และเลือกเก็บข้อมูลล่าสุดตามลำดับ (`keep='last'`) ตามกติกาทางธุรกิจ จะเหลือคำสั่งซื้อทั้งหมด **750 แถว** (ลบแถวซ้ำออกไป 2 แถว ได้แก่ `ORD000056` และ `ORD000416`)

---

### **คำถามที่ 2: มีแถวที่ customer_id หรือ product_id ไม่พบใน Master Data อย่างละกี่แถว?**
- **คำตอบ:**
  - **customer_id ไม่พบใน Master Data (`customers_crm.csv`):** มีจำนวน **22 แถว** (ได้แก่รหัสที่ไม่มีในระบบ CRM: `C0161`, `C0162`, `C0163`, `C0164`, `C0165`)
  - **product_id ไม่พบใน Master Data (`product_master.xlsx`):** มีจำนวน **2 แถว** (ได้แก่รหัสสินค้าที่ไม่ปรากฏในฝ่ายจัดซื้อ: `P999`)

---

### **คำถามที่ 3: มียอดขายที่ใช้ได้จริงกี่ธุรกรรม และยอดขายสุทธิรวมเท่าใด?**
- **คำตอบ:**
  - **จำนวนธุรกรรมยอดขายที่ใช้ได้จริง (Fact Sales):** **660 ธุรกรรม** (จากคำสั่งซื้อ 750 แถว ถูกตัดออก 90 แถว เนื่องจากผิดกฎ Quantity $\le 0$, Unit Price หาย/ผิดปกติ, รหัสไม่พบใน Master, และ Payment Status ไม่ใช่ `PAID`)
  - **ยอดขายสุทธิรวม (Total Net Sales):** **10,224,044.08 บาท** (คำนวณจาก $\text{quantity} \times \text{unit\_price} \times (1 - \text{discount})$)

---

### **คำถามที่ 4: จังหวัดใดมียอดขายสุทธิสูงสุด?**
- **คำตอบ:**
  - **กรุงเทพมหานคร** มียอดขายสุทธิสูงสุด อยู่ที่ **2,612,955.87 บาท** (คิดเป็น 154 คำสั่งซื้อ ปริมาณสินค้ารวม 315 ชิ้น)
  - อันดับถัดไปคือ ขอนแก่น (2,031,942.50 บาท), ระยอง (1,523,169.34 บาท), เชียงใหม่ (1,477,338.45 บาท), ภูเก็ต (1,427,389.28 บาท), และ ชลบุรี (1,151,248.64 บาท)

---

### **คำถามที่ 5: หมวดสินค้าใดมียอดขายสุทธิสูงสุด?**
- **คำตอบ:**
  - หมวดสินค้า **Smartphone** มียอดขายสุทธิสูงสุด อยู่ที่ **3,092,117.34 บาท** (จำนวน 178 คำสั่งซื้อ)
  - อันดับถัดไปคือ Accessory (2,710,583.47 บาท), Notebook (2,221,495.27 บาท), และ Smart Home (2,199,848.00 บาท)

---

### **คำถามที่ 6: หากสลับลำดับ merge ก่อน cleaning ผลลัพธ์หรือความเชื่อมั่นของข้อมูลเปลี่ยนอย่างไร?**
- **คำตอบ:**
  หากนำข้อมูลดิบมา Merge ก่อน Clean จะส่งผลกระทบต่อผลลัพธ์และความเชื่อมั่นของข้อมูล 4 ประการสำคัญ:
  1. **เกิด Cartesian Product / ข้อมูลบวมเกินจริง (Cardinality Explosion):**  
     ในข้อมูล Customer และ Payments มีแถวซ้ำ (Duplicate Primary Key) เช่น ลูกค้า `C0012`, `C0045`, `C0088` ซ้ำใน CRM และ `ORD000101` ซ้ำใน Payments หาก Merge ก่อน Deduplicate จะเกิดปัญหา **Many-to-Many Fan-out** ทำให้ยอดขายและจำนวนแถวใน Fact Sales เบิ้ลซ้ำเกินจริง
  2. **เกิดปัญหา Unmatched Join จากข้อมูลไม่เป็นมาตรฐาน (Data Loss):**  
     ข้อมูล Customer มีช่องว่าง (Trailing/Leading Whitespace) และชื่อจังหวัดหลายภาษา (`Bangkok`, `กทม.`, `Chiang Mai`, `ขอนเเก่น`) หาก Merge หรือ Group By ก่อน Standardize จะทำให้ข้อมูลแตกเป็นคนละกลุ่ม และรหัสที่มีช่องว่างจะไม่สามารถจับคู่ Foreign Key ได้
  3. **ข้อผิดพลาดในการคำนวณยอดขาย (Type & Value Computation Failure):**  
     ข้อมูลในเดือน ก.พ. มี `discount_pct` เป็นข้อความ string (`'5%'`) และมีค่า `unit_price` เป็น Null หรือติดลบ หากไม่ Clean ก่อนคำนวณ จะทำให้เกิด `TypeError` ใน Python หรือยอดขายผิดพลาด
  4. **สูญเสียความสามารถในการตรวจสอบย้อนกลับ (Loss of Auditability & Lineage):**  
     การ Clean และ Validate ทีละขั้นตอนช่วยให้เราสามารถบันทึกข้อผิดพลาด (Root Cause) ลงใน `data_quality_report.csv` ได้อย่างเป็นระบบและชี้แจงผู้บริหารได้ว่าแถวใดถูกตัดออกด้วยเหตุผลใด
